# Basic RAG Starter

A minimal RAG baseline using minsearch, following the same structure we used in the module: a `search` function, a `build_prompt` function, an `llm` function, and a top-level `rag` function that chains them together.

Replace the example data with your own and adapt the instructions to match what your project should actually do.

In [1]:
import json

from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
from minsearch import Index

openai_client = OpenAI()

## 1. Load Your Data

Replace this example with your own data. Each document should be a dict with a few text fields you want to search over.

In [2]:
import json

with open('../data/jobs.json', 'r') as f:
    documents = json.load(f)


## 2. Build the Index

minsearch builds an in-memory TF-IDF index over the fields you specify.

In [3]:
index = Index(
    text_fields=["job_title", "company", "description", "requirements"],
    keyword_fields=["location", "seniority_level"],
)

index.fit(documents)

## 3. Define the RAG Functions

`search` retrieves relevant documents, `build_prompt` formats them into a prompt, `llm` calls the model, and `rag` chains all three.

In [4]:
def search(query):
    return index.search(query, num_results=5)

In [5]:
instructions = """
You are an expert career advisor and job matching assistant.
The user will provide their CV or profile, and possibly some preferences.
Your task is to review the provided job postings in the CONTEXT,
and perform a personalized gap analysis.
For each relevant job:
1. Provide a Match Score (e.g., 80%).
2. Explain why they are a good fit.
3. Identify clear skill gaps (missing or weak skills based on the job requirements).
4. Give actionable suggestions to improve their chances.
5. Provide 2-3 interview preparation questions tailored to the role.
""".strip()


def build_prompt(query, search_results):
    clean_results = []
    for doc in search_results:
        clean_results.append({
            "job_title": doc.get("job_title"),
            "company": doc.get("company"),
            "requirements": doc.get("requirements"),
            "description": doc.get("description"),
        })
    search_result_json = json.dumps(clean_results, indent=2)

    user_prompt = f"""
<USER_PROFILE>
{query}
</USER_PROFILE>

<AVAILABLE_JOBS>
{search_result_json}
</AVAILABLE_JOBS>
""".strip()

    return user_prompt

In [6]:
def llm(user_prompt, instructions=None, model='gpt-4o-mini'):
    messages = []

    if instructions is not None:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text

In [7]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, instructions)
    return answer

## 4. Try It Out

In [8]:
user_cv = """
I am a Python developer with 3 years of experience. 
I have worked extensively with Django, Flask, and PostgreSQL. 
I also have some frontend experience with basic HTML/CSS and vanilla JavaScript. 
I'm looking for a Mid-Level Backend or Fullstack Engineer role in London or Remote.
"""
print(rag(user_cv))

### Job 1: Backend Python Engineer at TechFlow Inc.

1. **Match Score:** 75%
2. **Why they are a good fit:**
   - You meet the core requirement of 3+ years in Python and have experience with Django and PostgreSQL.
   - Your backend experience aligns with the focus on building scalable APIs and microservices.

3. **Clear Skill Gaps:**
   - Experience with FastAPI (required).
   - Familiarity with Docker, Kubernetes, and AWS.
   - Go experience is a plus but not essential.

4. **Actionable Suggestions:**
   - Learn FastAPI through online courses or tutorials to get hands-on experience.
   - Familiarize yourself with Docker and Kubernetes; Docker's official documentation has great resources for beginners.
   - Consider gaining some experience with AWS, perhaps by deploying a small Django app on their platform.

5. **Interview Preparation Questions:**
   - Can you describe an experience where you built a RESTful API? What frameworks did you use, and what challenges did you face?
   - How d